# div-prune: two batches (2026-10-26)Two questions, in priority order.**Batch 1 — head-count probe.** The rejection said the contribution is small.The deeper problem is that the current setting (8 heads, 92k-param GAT, threecitation networks) cannot show any benefit from head diversity: with eightheads on this task the heads are massively redundant, so making them diversecan at best be harmless. This batch holds the total hidden width fixed at 64and varies only how it is split across heads (8x8, 4x16, 2x32). If diversitymatters anywhere, it is at H=2, where a baseline model wastes half itscapacity on two near-identical heads. Expect no difference at H=8 and acontroller advantage at H=2 — that interaction *is* the contribution.**Batch 2 — similarity-accuracy frontier.** Logs inter-head similarity at thefinal epoch for every policy (previously controller-only) so that policies canbe compared at matched similarity instead of at mismatched checkpoints.Run in order. Every experiment cell is resumable per (dataset, arm, seed,hidden_per_head); re-run the cell after a disconnect and it continues.

In [ ]:
# GPU check (expect a T4 or better)
!nvidia-smi

In [ ]:
# Install dependencies and pull the current code (contains the div_last logging)
!pip install -q torch-geometric hydra-core omegaconf tqdm scipy pandas

import os
if not os.path.isdir('/content/div-prune'):
    !git clone -q https://github.com/HuKejie/div-prune /content/div-prune
os.chdir('/content/div-prune')
!git log --oneline -1
!pip install -q -e . --no-deps
!ls

In [ ]:
# Smoke test: one probe config, 2 seeds, 5 epochs — verify div_last and the
# non-default head count both work
!python -m run.pipeline.train experiment=gat_cora experiment.regularizer=attention \
    experiment.lambda_schedule=controller model.num_heads=2 model.hidden_per_head=32 \
    experiment.runs=2 model.epochs=5 seed=0 output_dir=outputs_smoke

import pandas as pd
smoke = pd.read_csv('outputs_smoke/tables/results.csv')
print(smoke[['dataset', 'seed', 'hidden_per_head', 'lambda_schedule', 'div_last', 'test_acc']])
assert smoke['div_last'].notna().all(), "div_last missing — code is not up to date"
assert (smoke['hidden_per_head'] == 32).all(), "hidden_per_head override not applied"
print("OK: probe config runs and div_last is recorded")

In [ ]:
# === shared runner ===
import os
import subprocess
from datetime import datetime

import pandas as pd

CSV_PATH = "outputs_frontier/tables/results.csv"
OUT_DIR = "outputs_frontier"
SEEDS = list(range(10))  # paper protocol

# Total hidden width is held at 64 in every probe config, so num_heads is
# determined by hidden_per_head (64 // hidden_per_head). The CSV stores
# hidden_per_head, which is what the analysis keys on.
PROBE_CONFIGS = [
    # (num_heads, hidden_per_head)
    (8, 8),
    (4, 16),
    (2, 32),
]

# batch 2 arms: dense fixed-lambda sweep around the 0.7 budget plus baselines
FRONTIER_ARMS = [
    # (label, regularizer, lambda_schedule, lambda_fixed)
    ("baseline",   "none",      "fixed",      0.1),
    ("fixed0.5",   "attention", "fixed",      0.5),
    ("fixed1",     "attention", "fixed",      1.0),
    ("fixed2",     "attention", "fixed",      2.0),
    ("fixed3",     "attention", "fixed",      3.0),
    ("fixed5",     "attention", "fixed",      5.0),
    ("fixed10",    "attention", "fixed",      10.0),
    ("fixed20",    "attention", "fixed",      20.0),
    ("controller", "attention", "controller", 0.0),
    ("linear",     "attention", "linear",     0.0),
    ("kendall",    "attention", "kendall",    0.0),
    ("gradnorm",   "attention", "gradnorm",   0.0),
]

# batch 1 arms: only the two that matter for the head-count question
PROBE_ARMS = [
    ("baseline",   "none",      "fixed",      0.1),
    ("controller", "attention", "controller", 0.0),
]


def run(cmd):
    print("$", cmd, flush=True)
    return subprocess.run(cmd, shell=True, check=False).returncode


def plog(msg):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} {msg}"
    with open("progress_log.md", "a", encoding="utf-8") as f:
        f.write(line + "\n")
    print(f"[log] {line}", flush=True)


def _mask(df, ds, arm, hp):
    reg, sched, lam = arm[1], arm[2], arm[3]
    return (
        (df.dataset == ds)
        & (df.regularizer == reg)
        & (df.lambda_schedule == sched)
        & (df.lambda_fixed == lam)
        & (df.hidden_per_head == hp)
        & (df.prune_criterion.isna())
    )


def _n_done(ds, arm, hp):
    if not os.path.exists(CSV_PATH):
        return 0
    return len(pd.read_csv(CSV_PATH)[_mask(pd.read_csv(CSV_PATH), ds, arm, hp)].seed.unique())


def launch(ds, arm, hp, num_heads):
    # True if this (dataset, arm, head config) already has all seeds.
    done = _n_done(ds, arm, hp)
    if done >= len(SEEDS):
        print(f"[skip] {ds} {arm[0]} H={num_heads} complete", flush=True)
        return
    remaining = len(SEEDS) - done
    label, reg, sched, lam = arm
    rc = run(
        f"python -m run.pipeline.train experiment=gat_cora "
        f"experiment.dataset={ds} model.num_heads={num_heads} model.hidden_per_head={hp} "
        f"experiment.regularizer={reg} experiment.lambda_schedule={sched} "
        f"experiment.lambda_fixed={lam} experiment.runs={remaining} seed=0 "
        f"output_dir={OUT_DIR}"
    )
    plog(f"done rc={rc} {ds} {label} H={num_heads} runs={remaining}")


def run_probe(datasets, configs=PROBE_CONFIGS, arms=PROBE_ARMS):
    plog("START batch 1 (head-count probe)")
    for ds in datasets:
        for num_heads, hp in configs:
            for arm in arms:
                launch(ds, arm, hp, num_heads)
    plog("END batch 1")


def run_frontier(datasets, arms=FRONTIER_ARMS):
    plog("START batch 2 (frontier)")
    for ds in datasets:
        for arm in arms:
            launch(ds, arm, 8, 8)  # frontier uses the paper config: 8 heads x 8
    plog("END batch 2")

In [ ]:
# === BATCH 1 (run this first): head-count probe, Cora, 10 seeds ===
# 3 head configs x 2 arms x 10 seeds = 60 runs.
# Reads: does the controller help more as heads become scarcer?
run_probe(["cora"])

In [ ]:
# === batch 1 analysis: the interaction is the result ===
# The claim to look for: no difference at H=8, controller advantage at H=2.
# If delta grows as H shrinks, diversity has a measurable payoff and the paper
# has a contribution instead of a non-claim.
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

df = pd.read_csv(CSV_PATH)
df = df[df.prune_criterion.isna()].copy()


def paired(ds, hp):
    d = df[(df.dataset == ds) & (df.hidden_per_head == hp)]
    b = d[d.regularizer == "none"].set_index("seed").test_acc
    c = d[d.lambda_schedule == "controller"].set_index("seed").test_acc
    common = b.index.intersection(c.index)
    if len(common) < 5:
        return None
    diff = (c[common].values - b[common].values) * 100
    try:
        p = wilcoxon(c[common].values, b[common].values).pvalue
    except Exception:
        p = float("nan")
    return {
        "H": 64 // hp,
        "n": len(common),
        "baseline_pp": round(b[common].mean() * 100, 2),
        "controller_pp": round(c[common].mean() * 100, 2),
        "delta_pp": round(diff.mean(), 2),
        "ci95": [round(diff.mean() - 1.96 * diff.std(ddof=1) / np.sqrt(len(diff)), 2),
                 round(diff.mean() + 1.96 * diff.std(ddof=1) / np.sqrt(len(diff)), 2)],
        "wilcoxon_p": round(p, 4) if p == p else None,
        "ctrl_div_last": round(d[d.lambda_schedule == "controller"].div_last.mean(), 3),
        "base_div_last": round(d[d.regularizer == "none"].div_last.mean(), 3),
    }


for ds in sorted(df.dataset.unique()):
    print(f"\n===== {ds.upper()} — head-count probe =====")
    rows = [r for r in (paired(ds, hp) for hp in (32, 16, 8)) if r]
    if rows:
        print(pd.DataFrame(rows).to_string(index=False))
        print("\nread: delta_pp should grow as H shrinks (last row = H=2)")

In [ ]:
# === BATCH 2: similarity-accuracy frontier, Cora ===
# Only needed for the "matched similarity" comparison; batch 1 answers the
# contribution question. Run it once batch 1 looks usable.
run_frontier(["cora"])

In [ ]:
# === batch 2 analysis: frontier table + similarity-matched comparison ===
def arm_of(r):
    if r.regularizer == "none":
        return "baseline"
    if r.lambda_schedule == "fixed":
        return f"fixed{r.lambda_fixed:g}"
    return r.lambda_schedule


fdf = df[df.hidden_per_head == 8].copy()
fdf["arm"] = fdf.apply(arm_of, axis=1)


def frontier(ds):
    d = fdf[fdf.dataset == ds]
    g = (
        d.groupby("arm")
        .agg(
            n=("test_acc", "size"),
            acc=("test_acc", "mean"),
            acc_sd=("test_acc", "std"),
            div_last=("div_last", "mean"),
            div_last_sd=("div_last", "std"),
            div_best=("div_final", "mean"),
            lam_last=("lambda_last", "mean"),
        )
        .reset_index()
    )
    g["acc_pp"] = (g["acc"] * 100).round(2)
    g["acc_sd_pp"] = (g["acc_sd"] * 100).round(2)
    g["div_last"] = g["div_last"].round(3)
    return g[["arm", "n", "acc_pp", "acc_sd_pp", "div_last", "div_last_sd",
              "div_best", "lam_last"]].sort_values("div_last")


def matched_compare(ds):
    d = fdf[fdf.dataset == ds]
    ctrl = d[d.arm == "controller"]
    if ctrl.empty:
        return None
    ctrl_div = ctrl.div_last.mean()
    fixed = d[d.arm.str.startswith("fixed") & (d.arm != "baseline")]
    if fixed.empty:
        return None
    best_arm = (fixed.groupby("arm").div_last.mean() - ctrl_div).abs().idxmin()
    m = fixed[fixed.arm == best_arm]
    a = ctrl.set_index("seed").test_acc
    b = m.set_index("seed").test_acc
    common = a.index.intersection(b.index)
    if len(common) < 5:
        return None
    diff = (a[common].values - b[common].values) * 100
    try:
        p = wilcoxon(a[common].values, b[common].values).pvalue
    except Exception:
        p = float("nan")
    return {
        "matched_arm": best_arm,
        "ctrl_div_last": round(ctrl_div, 3),
        "matched_div_last": round(m.div_last.mean(), 3),
        "delta_pp": round(diff.mean(), 2),
        "wilcoxon_p": round(p, 4) if p == p else None,
        "n_pairs": len(common),
    }


for ds in sorted(fdf.dataset.unique()):
    print(f"\n===== {ds.upper()} frontier =====")
    print(frontier(ds).to_string(index=False))
    mc = matched_compare(ds)
    if mc:
        print("\nsimilarity-matched (controller vs closest fixed arm):")
        for k, v in mc.items():
            print(f"  {k}: {v}")

In [ ]:
# === frontier plot: accuracy vs late-phase similarity (upper-left is better) ===
import matplotlib.pyplot as plt

for ds in sorted(fdf.dataset.unique()):
    g = frontier(ds)
    fig, ax = plt.subplots(figsize=(6, 4))
    for _, r in g.iterrows():
        is_ctrl = r.arm == "controller"
        ax.errorbar(r.div_last, r.acc_pp, xerr=r.div_last_sd, yerr=r.acc_sd_pp,
                    fmt="o", color="crimson" if is_ctrl else "steelblue",
                    markersize=8 if is_ctrl else 5)
        ax.annotate(r.arm, (r.div_last, r.acc_pp), fontsize=8, xytext=(4, 4),
                    textcoords="offset points")
    ax.set_xlabel("late-phase inter-head similarity (final epoch)")
    ax.set_ylabel("test accuracy (%)")
    ax.set_title(f"{ds}: similarity-accuracy frontier")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# === OPTIONAL: extend both batches to Citeseer + Pubmed ===
# run_probe(["citeseer", "pubmed"])
# run_frontier(["citeseer", "pubmed"])

In [ ]:
# === export: copy results to Drive so they can be pulled back locally ===
from google.colab import drive

drive.mount('/content/drive')
!cp outputs_frontier/tables/results.csv "/content/drive/MyDrive/div-prune-frontier-results.csv"
!ls -la "/content/drive/MyDrive/div-prune-frontier-results.csv"
print("Download this file and drop it into the repo as outputs_frontier/tables/results.csv")